# 📊 Sales EDA Dashboard
**Author:** Jeet Kumar | [GitHub](https://github.com/JayZCrash2000)

End-to-end exploratory data analysis on a retail sales dataset. Covers data cleaning, trend analysis, product/region performance, and actionable business insights.

---

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab10')
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded ✅')

## 2. Generate Synthetic Sales Dataset

In [ ]:
np.random.seed(42)
n = 2000

categories   = ['Electronics', 'Clothing', 'Furniture', 'Grocery', 'Automotive']
regions      = ['North', 'South', 'East', 'West', 'Central']
segments     = ['Consumer', 'Corporate', 'Home Office']
ship_modes   = ['Standard', 'Second Class', 'First Class', 'Same Day']

dates = pd.date_range('2022-01-01', '2024-12-31', periods=n)
dates = np.sort(np.random.choice(dates, n, replace=False))

category  = np.random.choice(categories, n, p=[0.30, 0.25, 0.15, 0.20, 0.10])
region    = np.random.choice(regions, n)
segment   = np.random.choice(segments, n, p=[0.52, 0.30, 0.18])
ship_mode = np.random.choice(ship_modes, n, p=[0.60, 0.20, 0.15, 0.05])

# Base price by category
base_price = {'Electronics': 450, 'Clothing': 80, 'Furniture': 300, 'Grocery': 40, 'Automotive': 220}
price = np.array([base_price[c] * np.random.uniform(0.5, 2.5) for c in category])
quantity = np.random.randint(1, 10, n)
discount = np.random.choice([0, 0.05, 0.10, 0.15, 0.20, 0.30], n, p=[0.40, 0.20, 0.15, 0.10, 0.10, 0.05])
sales    = price * quantity * (1 - discount)
profit   = sales * np.random.uniform(0.05, 0.35, n) * (1 - discount * 1.5)
profit   = np.clip(profit, -sales * 0.1, sales * 0.4)

df = pd.DataFrame({
    'order_date': pd.to_datetime(dates),
    'category': category,
    'region': region,
    'segment': segment,
    'ship_mode': ship_mode,
    'quantity': quantity,
    'unit_price': price.round(2),
    'discount': discount,
    'sales': sales.round(2),
    'profit': profit.round(2)
})

df['year']       = df['order_date'].dt.year
df['month']      = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.strftime('%b')
df['quarter']    = df['order_date'].dt.quarter
df['profit_margin'] = (df['profit'] / df['sales'] * 100).round(2)

print(f'Dataset shape: {df.shape}')
df.head()

## 3. Data Quality Check

In [ ]:
print('=== DATA OVERVIEW ===')
print(f'Shape       : {df.shape}')
print(f'Date range  : {df["order_date"].min().date()} → {df["order_date"].max().date()}')
print(f'\nNull values:\n{df.isnull().sum()}')
print(f'\nDuplicates  : {df.duplicated().sum()}')
print(f'\nTotal Sales : ₹{df["sales"].sum():,.0f}')
print(f'Total Profit: ₹{df["profit"].sum():,.0f}')
print(f'Avg Margin  : {df["profit_margin"].mean():.1f}%')

## 4. Sales Trend Analysis

In [ ]:
# Monthly sales trend
monthly = df.groupby(df['order_date'].dt.to_period('M')).agg(
    sales=('sales', 'sum'),
    profit=('profit', 'sum'),
    orders=('sales', 'count')
).reset_index()
monthly['order_date'] = monthly['order_date'].dt.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Sales & Profit Trends (2022–2024)', fontsize=15, fontweight='bold')

axes[0].plot(monthly['order_date'], monthly['sales'], color='steelblue', lw=2)
axes[0].fill_between(monthly['order_date'], monthly['sales'], alpha=0.15, color='steelblue')
axes[0].set_title('Monthly Sales Revenue')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

profit_colors = ['#2ecc71' if p >= 0 else '#e74c3c' for p in monthly['profit']]
axes[1].bar(monthly['order_date'], monthly['profit'], color=profit_colors, width=20)
axes[1].set_title('Monthly Profit (Green = Positive, Red = Negative)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

plt.tight_layout()
plt.savefig('sales_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Category & Region Performance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Category & Region Performance', fontsize=15, fontweight='bold')

# Sales by category
cat_sales = df.groupby('category')['sales'].sum().sort_values(ascending=True)
axes[0,0].barh(cat_sales.index, cat_sales.values, color=sns.color_palette('tab10', len(cat_sales)))
axes[0,0].set_title('Total Sales by Category')
axes[0,0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e3:.0f}K'))

# Profit margin by category
cat_margin = df.groupby('category')['profit_margin'].mean().sort_values(ascending=True)
colors_margin = ['#e74c3c' if v < 0 else '#2ecc71' for v in cat_margin.values]
axes[0,1].barh(cat_margin.index, cat_margin.values, color=colors_margin)
axes[0,1].set_title('Avg Profit Margin by Category (%)')
axes[0,1].axvline(0, color='black', lw=0.8, linestyle='--')

# Sales by region
reg_sales = df.groupby('region')['sales'].sum()
axes[1,0].pie(reg_sales.values, labels=reg_sales.index, autopct='%1.1f%%',
              colors=sns.color_palette('tab10', len(reg_sales)), startangle=90)
axes[1,0].set_title('Sales Share by Region')

# Sales by segment
seg = df.groupby('segment')[['sales', 'profit']].sum()
x = np.arange(len(seg))
w = 0.35
axes[1,1].bar(x - w/2, seg['sales'],  width=w, label='Sales',  color='steelblue')
axes[1,1].bar(x + w/2, seg['profit'], width=w, label='Profit', color='coral')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(seg.index)
axes[1,1].set_title('Sales vs Profit by Customer Segment')
axes[1,1].legend()
axes[1,1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e3:.0f}K'))

plt.tight_layout()
plt.savefig('category_region.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Discount Impact Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Discount Impact on Profitability', fontsize=14, fontweight='bold')

# Discount vs profit scatter
axes[0].scatter(df['discount'], df['profit_margin'], alpha=0.3, s=10, color='purple')
axes[0].set_xlabel('Discount Rate')
axes[0].set_ylabel('Profit Margin (%)')
axes[0].set_title('Discount vs Profit Margin')
axes[0].axhline(0, color='red', lw=1, linestyle='--')

# Avg profit margin by discount bucket
df['discount_bucket'] = pd.cut(df['discount'], bins=[-0.01, 0, 0.10, 0.20, 0.35],
                                labels=['No Discount', '1-10%', '11-20%', '21-30%'])
disc_margin = df.groupby('discount_bucket', observed=True)['profit_margin'].mean()
bar_colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in disc_margin.values]
axes[1].bar(disc_margin.index, disc_margin.values, color=bar_colors)
axes[1].set_title('Avg Profit Margin by Discount Level')
axes[1].set_xlabel('Discount Level')
axes[1].set_ylabel('Avg Profit Margin (%)')
axes[1].axhline(0, color='black', lw=0.8, linestyle='--')

plt.tight_layout()
plt.savefig('discount_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Quarterly YoY Comparison

In [ ]:
quarterly = df.groupby(['year', 'quarter'])['sales'].sum().unstack('year')

quarterly.plot(kind='bar', figsize=(10, 5), width=0.7,
               color=['#3498db', '#e67e22', '#2ecc71'])
plt.title('Quarterly Sales — Year over Year Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Quarter')
plt.ylabel('Sales Revenue')
plt.xticks(ticks=range(4), labels=['Q1', 'Q2', 'Q3', 'Q4'], rotation=0)
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e3:.0f}K'))
plt.legend(title='Year')
plt.tight_layout()
plt.savefig('quarterly_yoy.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Correlation Heatmap

In [ ]:
numeric_cols = ['quantity', 'unit_price', 'discount', 'sales', 'profit', 'profit_margin']
plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, square=True, linewidths=0.5)
plt.title('Numerical Feature Correlation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Executive Summary — Key Business Insights

### 📈 Revenue
- **Electronics** is the highest revenue category, driven by high unit prices
- **Q4 consistently outperforms** other quarters — strong seasonal demand
- Revenue shows a **upward YoY trend** from 2022 → 2024

### 💰 Profitability
- **Grocery** has the highest profit margin despite low unit prices — due to volume and low discounting
- **Discounts above 20% consistently destroy margin** — profit margin turns negative at high discount tiers
- **Corporate segment** generates the highest absolute profit

### 🌍 Regional Performance
- Sales are relatively evenly distributed across regions — no single region dominates
- Regional strategy should focus on **margin improvement**, not just revenue growth

### ⚠️ Risk Flags
- High discount rates (>20%) applied to Furniture and Automotive are eroding margins
- Some months show **negative profit** — likely linked to aggressive seasonal discounting

### 🔧 Recommendations
1. **Cap discounts at 15%** across all categories to protect margins
2. **Double down on Electronics + Corporate segment** — highest revenue + profit combination
3. Investigate **loss-making months** for root cause (discount spikes vs demand drops)
4. Expand **Same Day shipping** for high-margin Electronics orders to improve NPS